In [1]:
# Pregunta: ¿cuántas horas, millas y semanas de trabajo acumuló cada conductor?
# La respuesta permite identificar el nivel de actividad antes de relacionarlo con datos maestros en P520.
#
# lectura ──► mapper (clave, valor) ──► shuffle ──► reducer ──► escritura

import csv
from pathlib import Path

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)

SOURCE = ROOT / "data" / "timesheet.csv"
MAPPER_OUTPUT = ROOT / "temp" / "mapper_pairs.csv"
SHUFFLE_OUTPUT = ROOT / "temp" / "shuffled_pairs.csv"
RESULT = ROOT / "submission" / "driver_metrics.csv"

with SOURCE.open(encoding="utf-8", newline="") as file:
    timesheet = list(csv.DictReader(file))

len(timesheet), timesheet[:2]

(1768,
 [{'driverId': '10',
   'week': '1',
   'hours-logged': '70',
   'miles-logged': '3300'},
  {'driverId': '10',
   'week': '2',
   'hours-logged': '70',
   'miles-logged': '3300'}])

In [2]:
# Esta consulta SQL especifica el resultado que queremos obtener.
# No se ejecutará: construiremos el mismo resultado con un modelo clave–valor.
#
# SELECT
#     driverId,
#     SUM("hours-logged") AS total_hours,
#     SUM("miles-logged") AS total_miles,
#     COUNT(*) AS weeks
# FROM timesheet
# GROUP BY driverId;

def map_pairs(records):
    pairs = []

    for record in records:
        key = record["driverId"]
        value = (
            float(record["hours-logged"]),
            float(record["miles-logged"]),
            1,
        )
        pairs.append((key, value))

    return pairs

In [3]:
# El mapper transforma cada semana en una pareja cuya clave es el conductor.

mapped_pairs = map_pairs(timesheet)

with MAPPER_OUTPUT.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file, lineterminator="\n")
    writer.writerow(["driverId", "hours", "miles", "weeks"])
    writer.writerows((key, *value) for key, value in mapped_pairs)

mapped_pairs[:5]

[('10', (70.0, 3300.0, 1)),
 ('10', (70.0, 3300.0, 1)),
 ('10', (60.0, 2800.0, 1)),
 ('10', (70.0, 3100.0, 1)),
 ('10', (70.0, 3200.0, 1))]

In [4]:
# El shuffle reúne en una lista los valores que comparten la misma clave.

def group_by_key(pairs):
    grouped_values = {}

    for key, value in pairs:
        grouped_values.setdefault(key, []).append(value)

    return grouped_values

In [5]:
# Después del shuffle, cada conductor tiene todas sus semanas en el mismo grupo.

values_by_driver = group_by_key(mapped_pairs)

with SHUFFLE_OUTPUT.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file, lineterminator="\n")
    writer.writerow(["driverId", "hours", "miles", "weeks"])
    for key, values in sorted(values_by_driver.items(), key=lambda item: int(item[0])):
        writer.writerows((key, *value) for value in values)

first_driver = min(values_by_driver, key=int)
first_driver, values_by_driver[first_driver][:3]

('10', [(70.0, 3300.0, 1), (70.0, 3300.0, 1), (60.0, 2800.0, 1)])

In [6]:
# El reducer produce una sola medida agregada para cada clave.

def reduce_by_key(grouped_values):
    metrics = []

    for driver_id, values in sorted(grouped_values.items(), key=lambda item: int(item[0])):
        total_hours = sum(hours for hours, _, _ in values)
        total_miles = sum(miles for _, miles, _ in values)
        weeks = sum(week_count for _, _, week_count in values)
        metrics.append(
            {
                "driverId": driver_id,
                "total_hours": round(total_hours, 2),
                "total_miles": round(total_miles, 2),
                "weeks": weeks,
                "mean_hours": round(total_hours / weeks, 2),
            }
        )

    return metrics

In [7]:
# La salida reducida será el insumo lógico de la unión con datos maestros en P520.

driver_metrics = reduce_by_key(values_by_driver)

with RESULT.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(driver_metrics[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(driver_metrics)

driver_metrics[:5]

[{'driverId': '10',
  'total_hours': 3232.0,
  'total_miles': 147150.0,
  'weeks': 52,
  'mean_hours': 62.15},
 {'driverId': '11',
  'total_hours': 3642.0,
  'total_miles': 179300.0,
  'weeks': 52,
  'mean_hours': 70.04},
 {'driverId': '12',
  'total_hours': 2639.0,
  'total_miles': 135962.0,
  'weeks': 52,
  'mean_hours': 50.75},
 {'driverId': '13',
  'total_hours': 2727.0,
  'total_miles': 134126.0,
  'weeks': 52,
  'mean_hours': 52.44},
 {'driverId': '14',
  'total_hours': 2781.0,
  'total_miles': 136624.0,
  'weeks': 52,
  'mean_hours': 53.48}]